# PhysAR-IDS -- NB4e (v6): classifier F1 under the PPO attack (full test set, per class)

**Why.** NB4 / NB4b / NB4c / NB4d store one number per attack: the weighted F1 on the *attack rows only*, with no clean reference on the same rows. That cannot show what an attack does to the **whole detector**: benign rows are excluded, so the cost of attack traffic being labelled benign is invisible, and nothing is reported per class. This notebook re-scores every saved PPO agent (and a random-search baseline at the same budget) and records, for each method and budget:

* **weighted F1 and macro F1 on the full test set** (97,463 rows: the benign rows keep their clean predictions, the 81,718 attack rows are replaced by the oracle's labels after the attack);
* **per-class F1 and recall**, and **benign precision** (the share of rows labelled benign that really are benign: it falls as attacks hide there);
* the clean (no attack) reference on exactly the same rows and the drop from it, with a paired test over the 5 seeds.

**No training.** It loads the agents saved by NB4 (0.1 sigma budget, unconstrained), NB4b (0.2 / 0.3 / 0.5 / 1.0 sigma) and NB4c (agents retrained with the training-range clip); a missing file is skipped with a message. Evaluation is the same batched code as NB4 / NB4c, and the per-class evasion it produces is checked against the values NB4c stored (difference should be 0). Runtime: a few minutes. NB4d (physical channel) now writes the same full-test metrics itself.

**Definitions.** The attacker perturbs only attack-class rows, so benign predictions never change. An attack row counts as "after the attack" in the state the agent leaves it after at most 10 queries (evaded rows stop at the first query that returns benign). F1 uses sklearn's weighted average over the six classes (support-weighted) and macro (unweighted).

## 0. Setup

In [ ]:
import os, sys, random, time, json, hashlib, shutil, platform, warnings
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
try:
    from sb3_contrib import RecurrentPPO
    from stable_baselines3.common.callbacks import CheckpointCallback
except ImportError as e:
    raise ImportError('Install in a terminal (not in a cell): pip install stable-baselines3 sb3-contrib gymnasium --no-deps') from e
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False

# ---- PPO / environment settings (unchanged from the reference design) ----
EPS_BOUND = 0.3; MAX_STEPS = 10
EVASION_REWARD = 10.0; STEP_PENALTY_SCALE = 0.01; TIMEOUT_PENALTY = -1.0
TOTAL_TIMESTEPS = 50000; CKPT_FREQ = 5000
EPS_TOTAL_BUDGET = 0.1
CLIP_BUDGETS = {'1.0': 1.0, 'unconstrained': None}     # budgets at which agents are RETRAINED with range clipping (stage B)
RETRAIN = False                                         # False = stage A only (re-score saved agents with clipping; minutes)
SAVED_AGENTS = {'0.1': ('ppo_attack2_budgeted_{v}_seed{s}', 0.1), '0.2': ('ppo_attack2_budget0p2_{v}_seed{s}', 0.2), '0.3': ('ppo_attack2_budget0p3_{v}_seed{s}', 0.3),
                '0.5': ('ppo_attack2_budget0p5_{v}_seed{s}', 0.5), '1.0': ('ppo_attack2_budget1p0_{v}_seed{s}', 1.0), 'unconstrained': ('ppo_attack2_unconstrained_{v}_seed{s}', None)}
EVAL_CHUNK = 8192

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} fit backend: {BACKEND_NAME}')
import sb3_contrib, stable_baselines3
print(f'sb3-contrib {sb3_contrib.__version__} | stable-baselines3 {stable_baselines3.__version__} | gymnasium {gym.__version__}')
print('Working directory:', os.getcwd())

## 1. Load v6 data and the NB2 tuned configuration

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

atk_tr = y_train != 0; X_atk_train, y_atk_train = X_train[atk_tr], y_train[atk_tr]
atk_te = y_test != 0;  X_atk_test,  y_atk_test  = X_test[atk_te],  y_test[atk_te]
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
print(f'Features {N_FEAT} | attacker-controlled (cyber) {N_CYBER} | training pool {X_atk_train.shape} | evaluation rows {X_atk_test.shape} in {len(np.unique(test_burst))} attack bursts')
print('Training pool class counts:', np.bincount(y_atk_train)[1:], '| evaluation class counts:', np.bincount(y_atk_test)[1:])
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

## 2. Oracle, environment and batched evaluator (NB4 code + the range clip)

In [ ]:
class Oracle:
    """Hard-label black-box oracle over the fitted XGBoost classifier."""
    def __init__(self, clf): self.clf = clf; self.n_queries = 0
    def predict(self, X):
        X = np.ascontiguousarray(X, dtype=np.float32); self.n_queries += len(X)
        return self.clf.predict(X)

def apply_action(x_cur, x_orig, action, eps_bound=EPS_BOUND, eps_total=None, range_clip=False):
    """In place on x_cur[:, CYBER_IDX]; returns the applied (post-clamp) perturbation."""
    a = np.clip(action, -eps_bound, eps_bound).astype(np.float32)
    old = x_cur[:, CYBER_IDX].copy(); prop = old + a
    if eps_total is not None:
        o = x_orig[:, CYBER_IDX]; prop = o + np.clip(prop - o, -eps_total, eps_total)
    if range_clip:   # never leave the observed training range [min, max] (a value already outside it may stay where it was, but may not go further out)
        prop = np.clip(prop, np.minimum(FEAT_LO[CYBER_IDX], x_orig[:, CYBER_IDX]), np.maximum(FEAT_HI[CYBER_IDX], x_orig[:, CYBER_IDX]))
    x_cur[:, CYBER_IDX] = prop
    return (prop - old).astype(np.float32)

def build_obs(x_cur, last_applied, last_pred, step):
    oh = np.zeros((len(x_cur), 6), dtype=np.float32); oh[np.arange(len(x_cur)), last_pred] = 1.0
    return np.concatenate([x_cur.astype(np.float32), last_applied.astype(np.float32), oh, np.full((len(x_cur), 1), step / MAX_STEPS, dtype=np.float32)], 1)

class IDSEvasionEnv(gym.Env):
    metadata = {'render_modes': []}
    def __init__(self, X_pool, y_pool, oracle, eps_bound=EPS_BOUND, max_steps=MAX_STEPS, seed=None, eps_total_budget=None, range_clip=False):
        super().__init__(); self.range_clip = range_clip
        self.X_pool, self.y_pool, self.oracle = X_pool, y_pool, oracle
        self.eps_bound, self.max_steps, self.eps_total_budget = eps_bound, max_steps, eps_total_budget
        self.rng = np.random.RandomState(seed)
        self.action_space = spaces.Box(low=-eps_bound, high=eps_bound, shape=(N_CYBER,), dtype=np.float32)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(N_FEAT + N_CYBER + 6 + 1,), dtype=np.float32)
    def _obs(self): return build_obs(self.x_current[None], self.last_applied[None], np.array([self.last_pred]), self.step_count)[0]
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        idx = self.rng.randint(0, len(self.X_pool))
        self.x_orig = self.X_pool[idx].copy(); self.x_current = self.x_orig.copy(); self.y_true = self.y_pool[idx]
        self.step_count = 0; self.last_applied = np.zeros(N_CYBER, dtype=np.float32)
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); return self._obs(), {}
    def step(self, action):
        applied = apply_action(self.x_current[None], self.x_orig[None], np.asarray(action, dtype=np.float32)[None], self.eps_bound, self.eps_total_budget, self.range_clip)[0]
        self.last_applied = applied; self.step_count += 1
        self.last_pred = int(self.oracle.predict(self.x_current[None])[0]); pen = STEP_PENALTY_SCALE * float(np.sum(applied ** 2))
        terminated = truncated = False
        if self.last_pred == 0: reward = EVASION_REWARD - pen; terminated = True
        elif self.step_count >= self.max_steps: reward = TIMEOUT_PENALTY - pen; truncated = True
        else: reward = -pen
        return self._obs(), reward, terminated, truncated, {'evaded': self.last_pred == 0}

def run_batched(policy, X0, oracle, eps_total, mode='ppo', rng=None, range_clip=False):
    """Run one episode per row of X0 in parallel (10 steps max). mode: 'ppo' (deterministic policy) or 'random'."""
    N = len(X0); x_cur = X0.copy(); last_applied = np.zeros((N, N_CYBER), dtype=np.float32)
    last_pred = np.concatenate([oracle.predict(X0[i:i + 65536]) for i in range(0, N, 65536)]).astype(np.int64); clean_pred = last_pred.copy()
    active = np.ones(N, bool); evaded = np.zeros(N, bool); steps = np.zeros(N, np.int64)
    if mode == 'ppo': h = np.zeros((1, N, 64), np.float32); c = np.zeros((1, N, 64), np.float32)
    for step in range(MAX_STEPS):
        idx = np.where(active)[0]
        if len(idx) == 0: break
        obs = build_obs(x_cur[idx], last_applied[idx], last_pred[idx], step)
        if mode == 'ppo':
            act = np.zeros((len(idx), N_CYBER), np.float32)
            for s in range(0, len(idx), EVAL_CHUNK):
                sl = idx[s:s + EVAL_CHUNK]
                a, (hn, cn) = policy.predict(obs[s:s + EVAL_CHUNK], state=(h[:, sl], c[:, sl]), episode_start=np.full(len(sl), step == 0), deterministic=True)
                act[s:s + EVAL_CHUNK] = a; h[:, sl] = hn; c[:, sl] = cn
        else:
            act = rng.uniform(-EPS_BOUND, EPS_BOUND, size=(len(idx), N_CYBER)).astype(np.float32)
        sub = x_cur[idx]; applied = apply_action(sub, X0[idx], act, EPS_BOUND, eps_total, range_clip); x_cur[idx] = sub; last_applied[idx] = applied
        pred = np.concatenate([oracle.predict(sub[i:i + 65536]) for i in range(0, len(sub), 65536)]).astype(np.int64); last_pred[idx] = pred
        ev = pred == 0; evaded[idx[ev]] = True; steps[idx[ev]] = step + 1; active[idx[ev]] = False
    return {'evaded': evaded, 'steps': steps, 'x_final': x_cur, 'pred_final': last_pred, 'clean_pred': clean_pred}

def per_class(vec_bool):
    return np.array([vec_bool[y_atk_test == c].mean() for c in range(1, 6)])
def mean_steps(res):
    return np.array([res['steps'][(y_atk_test == c) & res['evaded']].mean() if ((y_atk_test == c) & res['evaded']).any() else np.nan for c in range(1, 6)])
def summarise(res):
    disp = np.abs(res['x_final'][:, CYBER_IDX] - X_atk_test[:, CYBER_IDX])
    _ex = lambda Z: np.maximum(np.maximum(FEAT_LO[CYBER_IDX] - Z[:, CYBER_IDX], Z[:, CYBER_IDX] - FEAT_HI[CYBER_IDX]), 0).max(1)
    exc = np.maximum(_ex(res['x_final']) - _ex(X_atk_test), 0)    # sigma ADDED beyond what the clean test row already had outside the training min/max
    return {'per_class_evasion': per_class(res['evaded']), 'overall_evasion': float(res['evaded'].mean()), 'mean_steps_to_evade': mean_steps(res),
            'mean_queries': float(1 + np.where(res['evaded'], res['steps'], MAX_STEPS).mean()),
            'attack_f1': float(f1_score(y_atk_test, res['pred_final'], average='weighted')),
            'max_abs_disp_mean': float(disp.max(1).mean()), 'max_abs_disp_max': float(disp.max()), 'mean_excess_beyond_train_range': float(exc.mean()), 'frac_rows_beyond_range_0.5sigma': float((exc > 0.5).mean()),
            'evaded': res['evaded'].copy()}
print('environment + evaluator defined; cyber features controlled:', N_CYBER)

## 3. Metrics on the full test set

In [ ]:
from sklearn.metrics import precision_score, recall_score
LAB = list(range(6))
def full_metrics(pred_atk, benign_pred):
    """Full-test metrics when the attack rows carry predictions `pred_atk` and benign rows keep `benign_pred`."""
    pred = np.empty(len(y_test), dtype=np.int64); pred[y_test == 0] = benign_pred; pred[atk_te] = pred_atk
    return {'weighted_f1': float(f1_score(y_test, pred, average='weighted')), 'macro_f1': float(f1_score(y_test, pred, average='macro')),
            'per_class_f1': f1_score(y_test, pred, average=None, labels=LAB), 'per_class_recall': recall_score(y_test, pred, average=None, labels=LAB),
            'benign_precision': float(precision_score(y_test, pred, labels=[0], average=None)[0]), 'attack_detected': float((pred_atk != 0).mean())}
print('full_metrics defined; full test rows', len(y_test), '| attack rows', int(atk_te.sum()))

## 4. Per-seed run

In [ ]:
AGENT_FILES = dict(SAVED_AGENTS)                                  # tag -> (file pattern, cumulative budget in sigma)
CLIP_AGENTS = {'1.0': ('ppo_attack2_clip_1p0_{v}_seed{s}', 1.0), 'unconstrained': ('ppo_attack2_clip_unconstrained_{v}_seed{s}', None)}
if SMOKE_TEST: pass
M = {}; EV = {}; INFO = {}; t_all = time.time()          # M[seed][(method, tag)] -> metrics ; EV[seed][(method, tag)] -> per-class evasion
for seed in SEEDS:
    pf = f'attack2e_seed{seed}_{PIPELINE_VERSION}.pkl'
    avail = tuple(sorted(f for pat, _ in list(AGENT_FILES.values()) + list(CLIP_AGENTS.values()) for f in [pat.format(v=PIPELINE_VERSION, s=seed) + '.zip'] if os.path.exists(f)))
    cfg_fp = repr((best_params, MAX_STEPS, seed, X_atk_test.shape, float(X_atk_test[:5].sum()), avail, 'nb4e-v1'))
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: M[seed], EV[seed], INFO[seed] = saved['M'], saved['EV'], saved['info']; print(f'seed {seed}: result file found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True)
    clf = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); clf.fit(X_train, y_train, verbose=False); clf.set_params(device='cpu', n_jobs=-1); oracle = Oracle(clf)
    clean_pred = clf.predict(X_test); benign_pred = clean_pred[y_test == 0]
    M[seed] = {('clean', '-'): full_metrics(clean_pred[atk_te], benign_pred)}; EV[seed] = {}
    INFO[seed] = {'clean_weighted_f1': M[seed][('clean', '-')]['weighted_f1']}
    print(f'  clean weighted F1 {INFO[seed]["clean_weighted_f1"]:.4f}')
    jobs = [('PPO', tag, pat, b, False) for tag, (pat, b) in AGENT_FILES.items()] + [('PPO_clip', tag, pat, b, True) for tag, (pat, b) in CLIP_AGENTS.items()]
    for method, tag, pat, b, clip in jobs:
        path = pat.format(v=PIPELINE_VERSION, s=seed) + '.zip'
        if not os.path.exists(path): print(f'  {path} not found -- skipped'); continue
        agent = RecurrentPPO.load(path); res = run_batched(agent, X_atk_test, oracle, b, mode='ppo', range_clip=clip)
        M[seed][(method, tag)] = full_metrics(res['pred_final'], benign_pred); EV[seed][(method, tag)] = per_class(res['evaded'])
        if method == 'PPO':
            rr = run_batched(None, X_atk_test, oracle, b, mode='random', rng=np.random.RandomState(seed + 500)); M[seed][('Random', tag)] = full_metrics(rr['pred_final'], benign_pred); EV[seed][('Random', tag)] = per_class(rr['evaded'])
        m = M[seed][(method, tag)]
        print(f'  {method:<9} {tag:>13}: weighted F1 {m["weighted_f1"]:.4f} | macro {m["macro_f1"]:.4f} | benign precision {m["benign_precision"]:.3f} | attacks detected {m["attack_detected"]:.3f}', flush=True)
    joblib.dump({'fp': cfg_fp, 'M': M[seed], 'EV': EV[seed], 'info': INFO[seed]}, pf)
    print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 5. Consistency check against the evasion rates stored by NB4c

In [ ]:
diffs = []
for seed in SEEDS:
    f = f'attack2c_seed{seed}_{PIPELINE_VERSION}.pkl'
    if not os.path.exists(f): continue
    Rc = joblib.load(f)['R']
    for (method, tag), ev in EV[seed].items():
        k = ('saved_noclip', tag) if method == 'PPO' else (('retrained_clip', tag) if method == 'PPO_clip' else None)
        if k is not None and k in Rc: diffs.append(float(np.abs(np.asarray(Rc[k]['per_class_evasion']) - ev).max()))
print(f'compared {len(diffs)} agent/seed evaluations with NB4c: max |difference in per-class evasion| = {max(diffs):.2e}' if diffs else 'attack2c result files not found -- check skipped')

## 6. Results: full-test F1 under the PPO attack (mean +/- std over seeds)

In [ ]:
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
keys = [('clean', '-')] + [(m, t) for t in ['0.1', '0.2', '0.3', '0.5', '1.0', 'unconstrained'] for m in ['PPO', 'Random']] + [('PPO_clip', t) for t in ['1.0', 'unconstrained']]
def have(key): return [s for s in SEEDS if key in M[s]]
def a(key, f): return np.stack([np.asarray(M[s][key][f]) for s in have(key)])
ms = lambda x: f'{x.mean():.4f} +/- {x.std(ddof=1) if len(x) > 1 else 0:.4f}'
rows = []
for key in keys:
    if not have(key): continue
    w, mc, bp, ad = a(key, 'weighted_f1'), a(key, 'macro_f1'), a(key, 'benign_precision'), a(key, 'attack_detected')
    cw = np.array([M[s][('clean', '-')]['weighted_f1'] for s in have(key)])
    row = {'method': key[0], 'budget (sigma)': key[1], 'seeds': len(w), 'weighted F1': ms(w), 'macro F1': ms(mc), 'drop vs clean': f'{(cw - w).mean():+.4f}', 'benign precision': f'{bp.mean():.3f}', 'attack rows labelled non-benign': f'{ad.mean():.3f}'}
    if key[0] != 'clean' and len(w) > 1:
        d = cw - w; row['p (paired vs clean)'] = f'{stats.ttest_rel(cw, w).pvalue:.4f}' if d.std(ddof=1) > 0 else 'n/a'
    rows.append(row)
f1_table = pd.DataFrame(rows); pd.set_option('display.width', 300); print(f1_table.to_string(index=False))

## 7. Per-class F1 and recall under the attack

F1 mixes precision and recall, so a class's F1 can move for reasons other than its own rows evading (rows of other attack classes relabelled to it lower its precision; rows it previously lost to another label can come back). Read recall for "is this class still detected as itself" and the "attack rows labelled non-benign" column of section 6 for "is the attack still seen at all".

In [ ]:
pc_rows, rc_rows = [], []
for key in keys:
    if not have(key): continue
    f1, rc = a(key, 'per_class_f1'), a(key, 'per_class_recall')
    r1 = {'method': key[0], 'budget': key[1]}; r1.update({c: f'{f1[:, i].mean():.3f} +/- {f1[:, i].std(ddof=1) if len(f1) > 1 else 0:.3f}' for i, c in enumerate(class_names)}); pc_rows.append(r1)
    r2 = {'method': key[0], 'budget': key[1]}; r2.update({c: f'{rc[:, i].mean():.3f}' for i, c in enumerate(class_names)}); rc_rows.append(r2)
pcf1_table, pcrec_table = pd.DataFrame(pc_rows), pd.DataFrame(rc_rows)
print('Per-class F1:'); print(pcf1_table.to_string(index=False)); print('\nPer-class recall:'); print(pcrec_table.to_string(index=False))

## 8. PPO versus random search, paired over seeds (full-test weighted F1; a lower F1 = a stronger attack)

In [ ]:
stat_rows = []
for t in ['0.1', '0.2', '0.3', '0.5', '1.0', 'unconstrained']:
    if len(have(('PPO', t))) == len(SEEDS) and len(have(('Random', t))) == len(SEEDS) and len(SEEDS) > 1:
        p, r = a(('PPO', t), 'weighted_f1'), a(('Random', t), 'weighted_f1'); d = p - r
        stat_rows.append({'budget': t, 'PPO F1': f'{p.mean():.4f}', 'random F1': f'{r.mean():.4f}', 'PPO minus random': f'{d.mean():+.4f}', 'p (paired t)': f'{stats.ttest_rel(p, r).pvalue:.4f}' if d.std(ddof=1) > 0 else 'n/a'})
stat_table = pd.DataFrame(stat_rows); print(stat_table.to_string(index=False) if len(stat_table) else 'incomplete seeds')

## 9. Figure and save

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8)); tags = ['0.1', '0.2', '0.3', '0.5', '1.0']; xs = np.arange(len(tags))
for meth, mk in [('PPO', 'o-'), ('Random', 's--')]:
    ys = [a((meth, t), 'weighted_f1').mean() if have((meth, t)) else np.nan for t in tags]; es = [a((meth, t), 'weighted_f1').std(ddof=1) if len(have((meth, t))) > 1 else 0 for t in tags]; ax[0].errorbar(xs, ys, yerr=es, fmt=mk, capsize=3, label=meth)
if have(('clean', '-')): ax[0].axhline(a(('clean', '-'), 'weighted_f1').mean(), color='k', ls=':', label='clean')
if have(('PPO', 'unconstrained')): ax[0].axhline(a(('PPO', 'unconstrained'), 'weighted_f1').mean(), color='r', ls=':', label='PPO unconstrained')
ax[0].set_xticks(xs); ax[0].set_xticklabels(tags); ax[0].set_xlabel('cumulative cyber budget (sigma)'); ax[0].set_ylabel('full-test weighted F1'); ax[0].set_title('Detector F1 under the PPO attack'); ax[0].legend(fontsize=8); ax[0].grid(alpha=.3)
sel = [k for k in [('clean', '-'), ('PPO', '0.1'), ('PPO', '0.5'), ('PPO', '1.0'), ('PPO', 'unconstrained')] if have(k)]; w = 0.8 / max(1, len(sel))
for j, k in enumerate(sel): f1 = a(k, 'per_class_f1'); ax[1].bar(np.arange(6) + (j - (len(sel) - 1) / 2) * w, f1.mean(0), w, label=f'{k[0]} {k[1]}')
ax[1].set_xticks(np.arange(6)); ax[1].set_xticklabels(class_names, rotation=20); ax[1].set_ylabel('F1'); ax[1].set_title('Per-class F1'); ax[1].legend(fontsize=8); ax[1].grid(alpha=.3, axis='y')
plt.tight_layout(); plt.savefig(f'attack2e_f1_under_attack_{PIPELINE_VERSION}.png', dpi=150); plt.show()

joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'class_names': class_names, 'f1_table': f1_table, 'per_class_f1_table': pcf1_table, 'per_class_recall_table': pcrec_table, 'stat_table': stat_table,
             'per_seed': {s: {f'{k[0]}|{k[1]}': v for k, v in M[s].items()} for s in SEEDS}, 'info': INFO}, f'attack2e_results_{PIPELINE_VERSION}.pkl')
f1_table.to_csv(f'attack2e_f1_table_{PIPELINE_VERSION}.csv', index=False); pcf1_table.to_csv(f'attack2e_per_class_f1_{PIPELINE_VERSION}.csv', index=False)
pcrec_table.to_csv(f'attack2e_per_class_recall_{PIPELINE_VERSION}.csv', index=False); stat_table.to_csv(f'attack2e_ppo_vs_random_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB4e', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'sb3_contrib': sb3_contrib.__version__, 'torch': torch.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb4e_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'attack2e_results_{PIPELINE_VERSION}.pkl', f'attack2e_f1_table_{PIPELINE_VERSION}.csv', f'attack2e_per_class_f1_{PIPELINE_VERSION}.csv', f'attack2e_per_class_recall_{PIPELINE_VERSION}.csv', f'attack2e_ppo_vs_random_{PIPELINE_VERSION}.csv', f'attack2e_f1_under_attack_{PIPELINE_VERSION}.png', f'nb4e_run_manifest_{PIPELINE_VERSION}.json']:
    print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)